# Notebook 21 — Why is the M1 held-out monocyte population not one clean blob?

Switching M2 → M1 (dropping the generic-monocyte holdout, keeping only **non-classical monocyte `CL:0000875`**) raised R² but did *not* tighten MMD. The hypothesis: the held-out population is *mostly* homogeneous (a central blob) but has scattered cells, and those scattered cells may be filterable noise (doublets / ambient / a distinct sub-state). This notebook digs into **only the held-out OOD non-classical monocytes**.

Plan (mentor's suggestion):
1. Embed *just* the held-out OOD monocytes (PCA → neighbors → UMAP + Leiden) — no atlas reference, so within-population structure is visible.
2. Color by **species**, **donor**, **tissue**, and Leiden cluster to see what separates the scattered cells from the blob.
3. Flag the scattered cells (distance from the per-species centroid in PCA / minority Leiden clusters) and **characterize** them: species/donor composition + differential genes vs the blob.
4. Conclude whether there's a principled filter to apply in preprocessing (the train → prep feedback loop).

Known facts (from the M1 dataset): the `CL:0000875` cells are **100% lung** and span **32 donors**, 426 human + 426 mouse (≈426 in the OOD half). So tissue cannot explain the scatter — donor/biology is the axis to watch. Outputs go to `nb21_outputs/`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
from sklearn.model_selection import train_test_split

REPO = Path("/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT")
CELL_GPU = REPO / "cellot/cellot_gpu"
H5 = CELL_GPU / "datasets/speciesot-human-mouse-hvg/hvg_pearson_residuals_m1_v07.h5ad"
OUT = REPO / "speciesOT/baseline/analysis/nb21_outputs"
OUT.mkdir(parents=True, exist_ok=True)

HOLDOUT = "CL:0000875"                  # non-classical monocyte (M1 holdout)
CT = "cell_type_ontology_term_id"


def add_transport(adata, source="mouse", target="human", condition_col="condition"):
    m = {source: "source", target: "target"}
    out = adata.copy()
    out.obs = out.obs.copy()
    out.obs["transport"] = out.obs[condition_col].map(m)
    return out[out.obs["transport"].notna()].copy()


def split_toggle_ood(adata, groupby, holdout, key, mode, random_state=0, test_size=0.2):
    """Reproduce cellot's toggle_ood split so the OOD subset matches training/eval."""
    split = pd.Series(index=adata.obs_names, dtype=object)
    for _, idx in adata.obs.groupby(groupby, observed=False).groups.items():
        tr, te = train_test_split(idx, random_state=random_state, test_size=test_size)
        split.loc[tr] = "train"
        split.loc[te] = "test"
    hv = [holdout] if isinstance(holdout, str) else list(holdout)
    ood_ix = adata.obs_names[adata.obs[key].isin(hv)]
    a, b = train_test_split(ood_ix, random_state=random_state, test_size=0.5)
    if mode == "ood":
        split.loc[a] = "ignore"
        split.loc[b] = "ood"
    else:
        split.loc[a] = "train"
        split.loc[b] = "ood"
    adata.obs["split"] = split.astype("category")
    return adata


d = split_toggle_ood(add_transport(ad.read_h5ad(H5)), groupby="condition",
                     holdout=HOLDOUT, key=CT, mode="ood", random_state=0, test_size=0.2)

mono = d[(d.obs["split"] == "ood") & (d.obs[CT].astype(str) == HOLDOUT)].copy()
mono.X = mono.X.toarray() if hasattr(mono.X, "toarray") else np.asarray(mono.X)
mono.X = mono.X.astype(np.float32)

print("held-out OOD non-classical monocytes:", mono.shape)
print("\nby species:\n", mono.obs["condition"].value_counts())
print("\nby tissue:\n", mono.obs["tissue"].value_counts().head())
print("\nn donors:", mono.obs["donor_id"].nunique())

held-out OOD non-classical monocytes: (426, 1000)

by species:
 condition
mouse    219
human    207
Name: count, dtype: int64

by tissue:
 tissue
lung    426
Name: count, dtype: int64

n donors: 32


## 1. Embed only the held-out monocytes

PCA → neighbors → UMAP + Leiden on just these ~426 cells. Because there's no atlas reference diluting the view, the within-population structure (blob vs scatter, species split) is directly visible.

In [2]:
n_pcs = int(min(50, mono.n_obs - 1, mono.n_vars - 1))
sc.pp.pca(mono, n_comps=n_pcs)
sc.pp.neighbors(mono, n_neighbors=15, n_pcs=min(30, n_pcs))
sc.tl.umap(mono, min_dist=0.3, random_state=42)

try:
    sc.tl.leiden(mono, resolution=0.5, random_state=42, flavor="igraph", n_iterations=2, directed=False)
except Exception as e1:
    try:
        sc.tl.leiden(mono, resolution=0.5, random_state=42)
    except Exception as e2:
        print("leiden unavailable, using single cluster:", e2)
        mono.obs["leiden"] = "0"

print("UMAP done. leiden clusters:\n", mono.obs["leiden"].value_counts())

/n/home01/jzhou1125/miniforge3/envs/analysis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


UMAP done. leiden clusters:
 leiden
0    187
2    162
3     45
1     32
Name: count, dtype: int64


## 2. Color by species, donor, Leiden

Tissue is constant (all lung) so it can't explain the scatter — we look at species, donor, and unsupervised Leiden structure instead.

In [3]:
U = mono.obsm["X_umap"]
cmap = plt.get_cmap("tab20")
fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))

# species
for sp, c in [("human", "#d62728"), ("mouse", "#6baed6")]:
    m = (mono.obs["condition"].astype(str) == sp).values
    axes[0].scatter(U[m, 0], U[m, 1], s=16, c=c, alpha=0.8, edgecolors="none",
                    label=f"{sp} (n={int(m.sum())})")
axes[0].set_title("by species"); axes[0].legend(fontsize=8)

# donor
donors = mono.obs["donor_id"].astype(str)
for i, dn in enumerate(donors.value_counts().index.tolist()):
    m = (donors == dn).values
    axes[1].scatter(U[m, 0], U[m, 1], s=16, color=cmap(i % 20), alpha=0.8, edgecolors="none")
axes[1].set_title(f"by donor ({donors.nunique()} donors)")

# leiden
for i, cl in enumerate(sorted(mono.obs["leiden"].astype(str).unique(), key=lambda x: int(x) if x.isdigit() else x)):
    m = (mono.obs["leiden"].astype(str) == cl).values
    axes[2].scatter(U[m, 0], U[m, 1], s=16, color=cmap(i % 20), alpha=0.85, edgecolors="none",
                    label=f"cl{cl} (n={int(m.sum())})")
axes[2].set_title("by Leiden cluster"); axes[2].legend(fontsize=8)

for ax in axes:
    ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2")
fig.suptitle("M1 held-out non-classical monocytes (OOD) — embedding colorings", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT / "nb21_umap_colorings.png", dpi=150, bbox_inches="tight")
print("saved", OUT / "nb21_umap_colorings.png")
plt.show()

saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/nb21_outputs/nb21_umap_colorings.png


## 3. Flag the scattered cells

The two species form two clusters, so we measure scatter *within each species*: distance to the per-species centroid in PCA space, flagging the top 15% as "scattered". (Leiden minority clusters are an independent cross-check in the panel above.)

In [4]:
P = mono.obsm["X_pca"]
dist = np.zeros(mono.n_obs)
species = mono.obs["condition"].astype(str).values
for sp in np.unique(species):
    m = species == sp
    centroid = P[m].mean(axis=0)
    dist[m] = np.linalg.norm(P[m] - centroid, axis=1)
mono.obs["dist_to_centroid"] = dist

# top 15% most distant within each species -> "scattered"
thr = mono.obs.groupby("condition", observed=True)["dist_to_centroid"].transform(
    lambda s: s.quantile(0.85))
mono.obs["group"] = np.where(mono.obs["dist_to_centroid"] > thr, "scattered", "blob")
print(mono.obs["group"].value_counts())

fig, ax = plt.subplots(figsize=(7, 6))
for grp, c in [("blob", "#9ecae1"), ("scattered", "#e6550d")]:
    m = (mono.obs["group"].astype(str) == grp).values
    ax.scatter(U[m, 0], U[m, 1], s=20, c=c, alpha=0.85, edgecolors="none",
               label=f"{grp} (n={int(m.sum())})")
ax.legend()
ax.set_title("M1 OOD monocytes: blob vs scattered\n(top 15% PCA distance to per-species centroid)")
ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2")
fig.tight_layout()
fig.savefig(OUT / "nb21_outliers.png", dpi=150, bbox_inches="tight")
print("saved", OUT / "nb21_outliers.png")
plt.show()

group
blob         362
scattered     64
Name: count, dtype: int64


saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/nb21_outputs/nb21_outliers.png


## 4. Characterize the scattered cells

Are the scattered cells a few specific donors, or a distinct expression state? We look at species/donor composition and the genes differentially expressed in scattered vs blob (Wilcoxon).

In [5]:
# Species x group
print("species x group:\n",
      mono.obs.groupby(["group", "condition"], observed=True).size().unstack(fill_value=0))

# Which donors are over-represented among scattered cells?
frac = (mono.obs.groupby("donor_id", observed=True)["group"]
        .apply(lambda s: (s == "scattered").mean())
        .sort_values(ascending=False))
n_per_donor = mono.obs["donor_id"].value_counts()
donor_tbl = pd.DataFrame({"n_cells": n_per_donor, "scattered_frac": frac}).dropna()
donor_tbl = donor_tbl[donor_tbl["n_cells"] >= 3].sort_values("scattered_frac", ascending=False)
print("\ntop donors by scattered fraction (>=3 cells):\n", donor_tbl.head(10).round(3))

# Differential genes: scattered vs blob (per species to avoid species confound -> use human, the target)
deg_out = {}
for sp in ["human", "mouse"]:
    sub = mono[mono.obs["condition"].astype(str) == sp].copy()
    if sub.obs["group"].nunique() < 2:
        continue
    sc.tl.rank_genes_groups(sub, "group", groups=["scattered"], reference="blob", method="wilcoxon")
    df = sc.get.rank_genes_groups_df(sub, group="scattered")
    deg_out[sp] = df
    print(f"\n[{sp}] top genes UP in scattered vs blob:")
    print(df.sort_values("logfoldchanges", ascending=False)
          .head(12)[["names", "logfoldchanges", "pvals_adj"]].to_string(index=False))
    df.to_csv(OUT / f"nb21_scattered_vs_blob_degs_{sp}.csv", index=False)

donor_tbl.to_csv(OUT / "nb21_donor_scattered_fraction.csv")
print("\nsaved DEG + donor tables to", OUT)

species x group:
 condition  human  mouse
group                  
blob         176    186
scattered     31     33

top donors by scattered fraction (>=3 cells):
           n_cells  scattered_frac
donor_id                         
18_46_F         6           1.000
18_45_M         3           1.000
24_61_M         5           1.000
24_59_M         3           1.000
TSP1           37           0.351
TSP14          18           0.333
TSP25           6           0.333
TSP2          146           0.068
30-M-3         26           0.038
3-M-5/6         3           0.000



[human] top genes UP in scattered vs blob:
          names  logfoldchanges  pvals_adj
ENSG00000169306       25.766178   1.000000
ENSG00000131771       25.766178   1.000000
ENSG00000187513       25.696587   1.000000
ENSG00000030304       25.458706   1.000000
ENSG00000204385       25.318184   1.000000
ENSG00000136546       25.032515   1.000000
ENSG00000169071       24.844269   1.000000
ENSG00000011465       21.531736   0.907781
ENSG00000007908       20.538504   1.000000
ENSG00000171848       19.562769   1.000000
ENSG00000143801       19.322668   1.000000
ENSG00000091513       19.065905   1.000000

[mouse] top genes UP in scattered vs blob:
          names  logfoldchanges  pvals_adj
ENSG00000148773       27.608910   0.827587
ENSG00000168350       27.015327   0.084709
ENSG00000185950       26.493355   1.000000
ENSG00000114737       25.914017   1.000000
ENSG00000077942       25.791630   1.000000
ENSG00000064787       25.567556   1.000000
ENSG00000138028       24.476442   1.000000
ENSG00000

## 5. Conclusion

> **Update — see §6:** the "donor-driven" signal below is really a **sequencing-assay artifact** (Smart-seq2 vs 10x). §6 is the definitive cause and revises the recommendation toward a hard assay filter.

What the run shows (426 OOD monocytes: 207 human + 219 mouse, **all lung**, 32 donors):

- **Two species blobs plus a genuinely detached sub-cluster.** The UMAP has the expected mouse and human clouds, but also a small cluster sitting far away at UMAP2 ≈ −20 (Leiden `cl1`, ~32 cells) that is cleanly separated from both blobs. This detached cluster — not the diffuse scatter — is the clearest "not part of the population" signal.
- **The diffuse scatter is donor-driven, not tissue-driven.** Tissue is constant (lung), so it explains nothing. Instead the scattered fraction concentrates in specific donors: four mouse donors (`18_46_F`, `18_45_M`, `24_61_M`, `24_59_M`) are **100% scattered** (small n, 3–6 cells each), and on the human side `TSP1` (35%), `TSP14` (33%), `TSP25` (33%) are elevated versus `TSP2` (7%, the dominant donor). That is a donor/batch fingerprint.
- **No coherent expression program.** The scattered-vs-blob DEGs have very large log-fold-changes but `pvals_adj ≈ 1` (driven by near-zero, on/off sparse genes in a handful of cells) — i.e. there is no clean biological substate distinguishing them, which is consistent with ambient/low-quality/batch rather than a real transitional cell type.

**Implications for preprocessing (the train → prep loop):**

1. **Defensible hard filter:** drop the detached Leiden `cl1` sub-cluster — it is separated from the population on every coloring and is not a single donor's coherent biology.
2. **Handle the donor-concentrated scatter as a covariate, not a deletion.** Removing whole donors (especially the small mouse donors that are 100% scattered) would be cherry-picking real-but-minority data; better to keep them and either down-weight or model donor as batch.
3. **Report floor/ceiling-normalized MMD** (Workstream A) so the irreducible donor heterogeneity is accounted for rather than mistaken for model error — this is likely *why* M1's MMD didn't tighten even though R² rose.

Artifacts for the prep step are in `nb21_outputs/`: `nb21_donor_scattered_fraction.csv`, `nb21_scattered_vs_blob_degs_{human,mouse}.csv`, and the two figures.

## 6. The actual cause: sequencing assay (Smart-seq2 contamination)

The donor signal in §4 turned out to be a proxy for **sequencing technology**. The `_v07.h5ad` `obs` doesn't carry assay, so we look it up from the source files (`sampled_{mouse,human}_shared.h5ad`, which have an `assay` column) by cell barcode. The intended assay filter (`mouse → 10x 3' v2`, `human → 10x 3' v3`, recorded in the spec's `assay_filter`) was **never enforced** when this dataset was built, so a Smart-seq2 minority slipped in.

In [6]:
SRC = {
    "mouse": "/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/data/tabula_muris/sampled_mouse_shared.h5ad",
    "human": "/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/data/tabula_sapiens/sampled_human_shared.h5ad",
}

# Look up assay from the source files (backed = obs only) by cell barcode.
assay = pd.Series(index=mono.obs_names, dtype=object)
for sp, p in SRC.items():
    s = ad.read_h5ad(p, backed="r")
    nm = mono.obs_names[mono.obs["condition"] == sp]
    inter = nm.intersection(s.obs_names)
    assay.loc[inter] = s.obs.loc[inter, "assay"].astype(str).values
mono.obs["assay"] = assay.values
mono.obs["tech"] = np.where(mono.obs["assay"].str.contains("Smart", na=False), "Smart-seq2", "10x")

print("assay x species:\n", pd.crosstab(mono.obs["assay"], mono.obs["condition"]))
print("\ntech x blob/scattered:\n", pd.crosstab(mono.obs["tech"], mono.obs["group"]))
print("\ntech x leiden cluster:\n", pd.crosstab(mono.obs["tech"], mono.obs["leiden"]))
_sc = mono.obs[mono.obs["group"] == "scattered"]
_bl = mono.obs[mono.obs["group"] == "blob"]
print(f"\nscattered cells that are Smart-seq2: {(_sc['tech']=='Smart-seq2').mean()*100:.0f}%")
print(f"blob cells that are Smart-seq2:      {(_bl['tech']=='Smart-seq2').mean()*100:.0f}%")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, key, title in [(axes[0], "assay", "by assay"), (axes[1], "tech", "by technology")]:
    for i, val in enumerate(sorted(mono.obs[key].dropna().astype(str).unique())):
        m = (mono.obs[key].astype(str) == val).values
        ax.scatter(U[m, 0], U[m, 1], s=16, color=cmap(i % 20), alpha=0.85, edgecolors="none",
                   label=f"{val} (n={int(m.sum())})")
    ax.set_title(title); ax.legend(fontsize=8)
    ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2")
fig.suptitle("M1 OOD monocytes colored by sequencing assay", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT / "nb21_assay.png", dpi=150, bbox_inches="tight")
print("saved", OUT / "nb21_assay.png")
plt.show()

assay x species:
 condition   human  mouse
assay                   
10x 3' v2       0    187
10x 3' v3     179      0
Smart-seq2     28     32

tech x blob/scattered:
 group       blob  scattered
tech                       
10x          351         15
Smart-seq2    11         49

tech x leiden cluster:
 leiden        0   1    2   3
tech                        
10x         187   0  162  17
Smart-seq2    0  32    0  28

scattered cells that are Smart-seq2: 77%
blob cells that are Smart-seq2:      3%


saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/nb21_outputs/nb21_assay.png


### Conclusion (revised): the scatter is an assay artifact, and the fix is a real filter

- The 426 cells are **not single-platform**: mouse = 187 × `10x 3' v2` **+ 32 × Smart-seq2**; human = 179 × `10x 3' v3` **+ 28 × Smart-seq2**.
- **~77% of the "scattered" cells are Smart-seq2**, vs ~3% of the blob. The detached Leiden `cl1` cluster is **100% Smart-seq2**.
- The "scattered donors" from §4 (TSP1, TSP14, ...) are simply the donors with high Smart-seq2 content; the clean dominant donor TSP2 is ~99% 10x v3.

So this is **not biology and not really donor** — it is the well-known platform gap (Smart-seq2 is plate-based, full-length, no UMIs; 10x is droplet, 3', UMI). The intended per-species assay filter was recorded in the spec but **never enforced** during prep.

**Action (supersedes §5):** enforce the assay filter as a *required* preprocessing treatment for all atlas prep — keep `mouse → 10x 3' v2`, `human → 10x 3' v3`, drop everything else (Smart-seq2, 5', etc.). This removes exactly the scatter + the detached cluster, leaves single-platform populations per species, and should tighten MMD without cherry-picking. Implemented in `speciesOT/hub/prep.py` (`_apply_assay_filter`) going forward.